In [1]:
import numpy as np
import pandas as pd
import sklearn
import warnings
import random
import os
import sys
import time
import copy
import gc
import glob
import shutil
import traceback
from datetime import datetime
from pathlib import Path

from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

print('scikit-learn:', sklearn.__version__)
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('Python:', sys.version)

scikit-learn: 1.7.2
numpy: 2.0.0
pandas: 2.2.2
Python: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:12:24) [GCC 11.2.0]


In [2]:
from autogluon.tabular import TabularPredictor
import autogluon
from autogluon.tabular import __version__
import xgboost
print('Autogluon:', autogluon.tabular.__version__)
print('XGBoost:', xgboost.__version__)

Autogluon: 1.5.0
XGBoost: 3.2.0


In [3]:
from series_parallel import SeriesParallelEnsemble, tree_to_string, tree_to_string_with_names

In [4]:
def log_exception(exception, name, run, file_path="error_log.txt"):
    """Log an exception to a text file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    error_traceback = traceback.format_exc()
    error_message = (
        f"\n    ===== ERROR {timestamp} =====\n"
        f"    Type: {type(exception).__name__}\n"
        f"    Dataset: {name}\n"
        f"    Run: {str(run)}\n"
        f"    Message: {str(exception)}\n"
        f"    Traceback:\n    {error_traceback}\n"
        f"    =============================\n"
    )
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)
    with open(file_path, "a", encoding="utf-8") as file:
        file.write(error_message)
    print(f"Exception logged to '{file_path}'")


def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)


def count_leaves(tree):
    """Count the number of leaves (models used) in a tuple-tree."""
    if isinstance(tree, int):
        return 1
    _, left, right = tree
    return count_leaves(left) + count_leaves(right)

In [5]:
class CachedProbaEstimator:
    """Lightweight estimator serving predict_proba from pre-computed caches.
    Indexed by identity of the array X (id()) to avoid expensive comparisons.
    """
    def __init__(self, name, proba_caches):
        self.name = name
        self._caches = proba_caches  # list of (X_ref, probas_array)

    def predict_proba(self, X):
        x_id = id(X)
        for X_ref, probas in self._caches:
            if id(X_ref) == x_id:
                return probas
        raise KeyError(f"No cached probabilities for X (id={x_id}) in model '{self.name}'")

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    def __str__(self):
        return self.name

    def __repr__(self):
        return f"CachedProbaEstimator('{self.name}')"

In [6]:
FOLDER_DATASETS = "datasets_multiclass/"

class dataset:
    def __init__(self, name_ds, metric=None, random_state=123,
                 ag_time_limit=600, sp_random_time_limit=60,
                 ag_full_time_limit=600,
                 run_sp_random_adaptive=True):
        self.name_file = name_ds
        self.random_state = random_state

        # Data splits
        self.train_X = self.train_Y = None  # T  (60%)
        self.val_X   = self.val_Y   = None  # V  (20%)
        self.test_X  = self.test_Y  = None  # t  (20%)

        # AutoGluon
        self.ag_time_limit = ag_time_limit
        self.ag_full_time_limit = ag_full_time_limit
        self.sp_random_time_limit = sp_random_time_limit
        self.run_sp_random_adaptive = run_sp_random_adaptive
        self.ag_predictor = None
        self.ag_predictor_full = None
        self.l1_model_names = []
        self.probas_train = {}
        self.probas_val = {}
        self.probas_test = {}
        self.cached_estimators = []

        # Metric
        if metric is None:
            self.metric = accuracy_score
        else:
            self.metric = metric

        self.dict_results = dict(name_dataset=name_ds, random_state=random_state)

    # ─────────────────────────────────────────────────────────
    # Load dataset  (60/20/20 stratified split)
    # ─────────────────────────────────────────────────────────
    def load_dataset(self):
        X = pd.read_csv(self.name_file)
        y = X['target_end']
        X = X.drop(columns='target_end')

        le = LabelEncoder()
        y_enc = le.fit_transform(y)
        self.label_encoder = le

        # 80/20 split -> train+val / test
        X_trainval, self.test_X, y_trainval, self.test_Y = train_test_split(
            X.values, y_enc, test_size=0.2,
            stratify=y_enc, random_state=self.random_state)

        # 75/25 of trainval -> train (60%) / val (20%)
        self.train_X, self.val_X, self.train_Y, self.val_Y = train_test_split(
            X_trainval, y_trainval, test_size=0.25,
            stratify=y_trainval, random_state=self.random_state)

        # Build DataFrames for AutoGluon (needs column names + target)
        col_names = [f"f{i}" for i in range(self.train_X.shape[1])]
        self._train_df = pd.DataFrame(self.train_X, columns=col_names)
        self._train_df['target_end'] = self.train_Y
        self._val_df = pd.DataFrame(self.val_X, columns=col_names)
        self._val_df['target_end'] = self.val_Y
        self._test_df = pd.DataFrame(self.test_X, columns=col_names)
        self._test_df['target_end'] = self.test_Y

    # ─────────────────────────────────────────────────────────
    # AutoGluon: train L1 base models (no stacking)
    # ─────────────────────────────────────────────────────────
    def train_autogluon_base_models(self):
        tic = time.time()
        ag_path = f'./ag_models/{Path(self.name_file).stem}_seed{self.random_state}/'
        self.ag_predictor = TabularPredictor(
            label='target_end',
            eval_metric='f1_macro',
            path=ag_path,
            verbosity=0,
        ).fit(
            self._train_df,
            tuning_data=self._val_df,
            use_bag_holdout=True,
            time_limit=self.ag_time_limit,
            num_stack_levels=0,
            num_bag_folds=0,
            fit_weighted_ensemble=True,
            presets='medium_quality',
            hyperparameters='default',
        )
        ag_fit_time = time.time() - tic

        # Extract L1 model names (exclude WeightedEnsemble)
        leaderboard = self.ag_predictor.leaderboard(silent=True)
        self.l1_model_names = [
            m for m in leaderboard[leaderboard['stack_level'] == 1]['model'].tolist()
            if 'WeightedEnsemble' not in m
        ]

        # Verify class label alignment
        ag_classes = list(self.ag_predictor.class_labels)
        expected_classes = list(range(len(ag_classes)))
        assert ag_classes == expected_classes, \
            f"Class label mismatch: AG={ag_classes}, expected={expected_classes}"

        # Cache predict_proba for all L1 models on all splits.
        # Some models may fail at predict_proba time due to library bugs
        # (e.g., XGBoost n_classes_ incompatibility). We skip those and
        # keep only the models that work.
        failed_models = []
        for m in self.l1_model_names:
            try:
                self.probas_train[m] = self.ag_predictor.predict_proba(
                    self._train_df.drop(columns='target_end'), model=m).values
                self.probas_val[m] = self.ag_predictor.predict_proba(
                    self._val_df.drop(columns='target_end'), model=m).values
                self.probas_test[m] = self.ag_predictor.predict_proba(
                    self._test_df.drop(columns='target_end'), model=m).values
            except Exception as e:
                print(f"    WARNING: skipping model '{m}' — predict_proba failed: {e}")
                failed_models.append(m)
                # Clean up any partial caches for this model
                self.probas_train.pop(m, None)
                self.probas_val.pop(m, None)
                self.probas_test.pop(m, None)

        # Remove failed models from the list
        if failed_models:
            self.l1_model_names = [m for m in self.l1_model_names if m not in failed_models]
            print(f"    {len(failed_models)} model(s) skipped, {len(self.l1_model_names)} remaining")

        # Build CachedProbaEstimator list
        self.cached_estimators = []
        for m in self.l1_model_names:
            caches = [
                (self.train_X, self.probas_train[m]),
                (self.val_X,   self.probas_val[m]),
                (self.test_X,  self.probas_test[m]),
            ]
            self.cached_estimators.append((m, CachedProbaEstimator(m, caches)))

        # Record results
        self.dict_results.update({
            'n_l1_models': len(self.l1_model_names),
            'n_l1_models_skipped': len(failed_models),
            'l1_model_names': str(self.l1_model_names),
            'ag_fit_time_sec': ag_fit_time,
        })
        print(f"  AutoGluon L1: {len(self.l1_model_names)} models in {ag_fit_time:.1f}s")

    # ─────────────────────────────────────────────────────────
    # Metrics for individual L1 models
    # ─────────────────────────────────────────────────────────
    def evaluate_individual_l1_models(self):
        best_val_score = -np.inf
        best_model_name = None
        for m in self.l1_model_names:
            preds_val  = np.argmax(self.probas_val[m], axis=1)
            preds_test = np.argmax(self.probas_test[m], axis=1)
            preds_train = np.argmax(self.probas_train[m], axis=1)
            score_train = self.metric(self.train_Y, preds_train)
            score_val   = self.metric(self.val_Y, preds_val)
            score_test  = self.metric(self.test_Y, preds_test)
            self.dict_results[f'l1_{m}_train_macro'] = score_train
            self.dict_results[f'l1_{m}_val_macro']   = score_val
            self.dict_results[f'l1_{m}_test_macro']  = score_test
            if score_val > best_val_score:
                best_val_score = score_val
                best_model_name = m
        self.dict_results['best_l1_model'] = best_model_name
        self.dict_results['best_l1_val_macro'] = best_val_score
        self.dict_results['best_l1_train_macro'] = self.dict_results[f'l1_{best_model_name}_train_macro']
        self.dict_results['best_l1_test_macro']  = self.dict_results[f'l1_{best_model_name}_test_macro']
        print(f"  Best L1: {best_model_name} (val={best_val_score:.4f}, "
              f"test={self.dict_results['best_l1_test_macro']:.4f})")

    # ─────────────────────────────────────────────────────────
    # WeightedEnsemble_L2 baseline
    # ─────────────────────────────────────────────────────────
    def evaluate_weighted_ensemble(self):
        lb = self.ag_predictor.leaderboard(silent=True)
        we_models = lb[lb['model'].str.contains('WeightedEnsemble')]['model'].tolist()
        if not we_models:
            print("  WeightedEnsemble_L2 not available")
            for s in ['train', 'val', 'test']:
                self.dict_results[f'we_l2_{s}_macro'] = np.nan
            return
        we_name = we_models[0]
        try:
            preds_train = self.ag_predictor.predict(
                self._train_df.drop(columns='target_end'), model=we_name)
            preds_val = self.ag_predictor.predict(
                self._val_df.drop(columns='target_end'), model=we_name)
            preds_test = self.ag_predictor.predict(
                self._test_df.drop(columns='target_end'), model=we_name)
            self.dict_results['we_l2_train_macro'] = self.metric(self.train_Y, preds_train)
            self.dict_results['we_l2_val_macro']   = self.metric(self.val_Y, preds_val)
            self.dict_results['we_l2_test_macro']  = self.metric(self.test_Y, preds_test)
            print(f"  WeightedEnsemble_L2: val={self.dict_results['we_l2_val_macro']:.4f}, "
                  f"test={self.dict_results['we_l2_test_macro']:.4f}")
        except Exception as e:
            print(f"  WARNING: WeightedEnsemble_L2 predict failed: {e}")
            for s in ['train', 'val', 'test']:
                self.dict_results[f'we_l2_{s}_macro'] = np.nan

    # ─────────────────────────────────────────────────────────
    # AutoGluon full pipeline baseline
    # ─────────────────────────────────────────────────────────
    def train_autogluon_full(self):
        tic = time.time()
        ag_full_path = f'./ag_full/{Path(self.name_file).stem}_seed{self.random_state}/'
        self.ag_predictor_full = TabularPredictor(
            label='target_end',
            eval_metric='f1_macro',
            path=ag_full_path,
            verbosity=0,
        ).fit(
            self._train_df,
            tuning_data=self._val_df,
            use_bag_holdout=True,
            time_limit=self.ag_full_time_limit,
            presets='best_quality',
        )
        ag_full_time = time.time() - tic

        preds_train = self.ag_predictor_full.predict(
            self._train_df.drop(columns='target_end'))
        preds_val = self.ag_predictor_full.predict(
            self._val_df.drop(columns='target_end'))
        preds_test = self.ag_predictor_full.predict(
            self._test_df.drop(columns='target_end'))
        self.dict_results['ag_full_train_macro'] = self.metric(self.train_Y, preds_train)
        self.dict_results['ag_full_val_macro']   = self.metric(self.val_Y, preds_val)
        self.dict_results['ag_full_test_macro']  = self.metric(self.test_Y, preds_test)
        self.dict_results['ag_full_time_sec']    = ag_full_time
        self.dict_results['ag_full_best_model']  = self.ag_predictor_full.model_best
        print(f"  AG full: val={self.dict_results['ag_full_val_macro']:.4f}, "
              f"test={self.dict_results['ag_full_test_macro']:.4f} "
              f"({ag_full_time:.1f}s)")

    # ─────────────────────────────────────────────────────────
    # Helper: compute metrics for an SP ensemble
    # ─────────────────────────────────────────────────────────
    def _compute_sp_metrics(self, prefix, sp_ensemble):
        preds_train = sp_ensemble.predict_multiclass(self.train_X)
        preds_val   = sp_ensemble.predict_multiclass(self.val_X)
        preds_test  = sp_ensemble.predict_multiclass(self.test_X)
        self.dict_results[f'{prefix}_train_macro'] = self.metric(self.train_Y, preds_train)
        self.dict_results[f'{prefix}_val_macro']   = self.metric(self.val_Y, preds_val)
        self.dict_results[f'{prefix}_test_macro']  = self.metric(self.test_Y, preds_test)

        # Tree info
        tree_tuple = sp_ensemble._best_tree_tuple
        n_models = count_leaves(tree_tuple)
        tree_str = tree_to_string(tree_tuple)
        model_names = [name for name, _ in sp_ensemble.estimators]
        tree_str_names = tree_to_string_with_names(tree_tuple, model_names)
        self.dict_results[f'{prefix}_n_models_used'] = n_models
        self.dict_results[f'{prefix}_tree'] = tree_str
        self.dict_results[f'{prefix}_tree_names'] = tree_str_names
        return n_models

    # ─────────────────────────────────────────────────────────
    # SP Greedy (allow_partial=True and False)
    # ─────────────────────────────────────────────────────────
    def train_sp_greedy(self):
        if len(self.cached_estimators) < 2:
            print("  SP greedy: need >= 2 models, skipping")
            for suffix in ['partial', 'full']:
                for split in ['train', 'val', 'test']:
                    self.dict_results[f'sp_greedy_{suffix}_{split}_macro'] = np.nan
            return

        for allow_partial, suffix in [(True, 'partial'), (False, 'full')]:
            tic = time.time()
            sp = SeriesParallelEnsemble(
                estimators=list(self.cached_estimators),
                mode='multiclass',
                metric=self.metric,
                strategy='greedy',
                random_state=self.random_state,
            )
            sp.fit(self.val_X, self.val_Y,
                   allow_partial=allow_partial,
                   beam_width=10,
                   n_restarts=5,
                   n_jobs=-1,
                   metric=self.metric)
            fit_time = time.time() - tic

            prefix = f'sp_greedy_{suffix}'
            n_models = self._compute_sp_metrics(prefix, sp)
            self.dict_results[f'{prefix}_time_sec'] = fit_time

            print(f"  SP greedy (allow_partial={allow_partial}): "
                  f"val={self.dict_results[f'{prefix}_val_macro']:.4f}, "
                  f"test={self.dict_results[f'{prefix}_test_macro']:.4f}, "
                  f"models={n_models}, time={fit_time:.1f}s")

    # ─────────────────────────────────────────────────────────
    # SP Random search (fixed time limit)
    # ─────────────────────────────────────────────────────────
    def train_sp_random(self):
        if len(self.cached_estimators) < 2:
            print("  SP random: need >= 2 models, skipping")
            for split in ['train', 'val', 'test']:
                self.dict_results[f'sp_random_{split}_macro'] = np.nan
            return

        tic = time.time()
        sp = SeriesParallelEnsemble(
            estimators=list(self.cached_estimators),
            mode='multiclass',
            metric=self.metric,
            strategy='random',
            random_state=self.random_state,
        )
        sp.fit(self.val_X, self.val_Y,
               time_limit_seconds=self.sp_random_time_limit,
               n_jobs=-1,
               metric=self.metric)
        fit_time = time.time() - tic

        n_models = self._compute_sp_metrics('sp_random', sp)
        n_evaluated = getattr(sp, '_random_search_n_evaluated', -1)
        self.dict_results['sp_random_n_evaluated'] = n_evaluated
        self.dict_results['sp_random_time_sec'] = fit_time

        print(f"  SP random: val={self.dict_results['sp_random_val_macro']:.4f}, "
              f"test={self.dict_results['sp_random_test_macro']:.4f}, "
              f"models={n_models}, evaluated={n_evaluated}, time={fit_time:.1f}s")

    # ─────────────────────────────────────────────────────────
    # SP Random search (adaptive time = AG_full - AG_L1)
    # ─────────────────────────────────────────────────────────
    def train_sp_random_adaptive(self):
        """SP Random with time budget = actual AG full time - actual AG L1 time.

        This gives SP random the same extra wall-clock time that AG full
        spent beyond the base model training, making the comparison fair:
        both methods start from the same L1 models and get the same
        additional compute budget.
        """
        prefix = 'sp_random_adaptive'

        if len(self.cached_estimators) < 2:
            print("  SP random adaptive: need >= 2 models, skipping")
            for split in ['train', 'val', 'test']:
                self.dict_results[f'{prefix}_{split}_macro'] = np.nan
            return

        ag_full_time = self.dict_results.get('ag_full_time_sec')
        ag_l1_time = self.dict_results.get('ag_fit_time_sec')

        if ag_full_time is None or ag_l1_time is None:
            print("  SP random adaptive: AG times not available, skipping")
            for split in ['train', 'val', 'test']:
                self.dict_results[f'{prefix}_{split}_macro'] = np.nan
            return

        adaptive_time = ag_full_time - ag_l1_time
        if adaptive_time <= 0:
            print(f"  SP random adaptive: computed budget <= 0 "
                  f"({ag_full_time:.1f} - {ag_l1_time:.1f} = {adaptive_time:.1f}), skipping")
            for split in ['train', 'val', 'test']:
                self.dict_results[f'{prefix}_{split}_macro'] = np.nan
            self.dict_results[f'{prefix}_time_budget'] = adaptive_time
            return

        print(f"  SP random adaptive budget: {adaptive_time:.1f}s "
              f"(AG_full={ag_full_time:.1f}s - AG_L1={ag_l1_time:.1f}s)")

        tic = time.time()
        sp = SeriesParallelEnsemble(
            estimators=list(self.cached_estimators),
            mode='multiclass',
            metric=self.metric,
            strategy='random',
            random_state=self.random_state,
        )
        sp.fit(self.val_X, self.val_Y,
               time_limit_seconds=adaptive_time,
               n_jobs=-1,
               metric=self.metric)
        fit_time = time.time() - tic

        n_models = self._compute_sp_metrics(prefix, sp)
        n_evaluated = getattr(sp, '_random_search_n_evaluated', -1)
        self.dict_results[f'{prefix}_n_evaluated'] = n_evaluated
        self.dict_results[f'{prefix}_time_sec'] = fit_time
        self.dict_results[f'{prefix}_time_budget'] = adaptive_time

        print(f"  SP random adaptive: val={self.dict_results[f'{prefix}_val_macro']:.4f}, "
              f"test={self.dict_results[f'{prefix}_test_macro']:.4f}, "
              f"models={n_models}, evaluated={n_evaluated}, time={fit_time:.1f}s")

    # ─────────────────────────────────────────────────────────
    # Cleanup
    # ─────────────────────────────────────────────────────────
    def cleanup(self):
        self.train_X = self.train_Y = None
        self.val_X = self.val_Y = None
        self.test_X = self.test_Y = None
        self._train_df = self._val_df = self._test_df = None
        self.probas_train = self.probas_val = self.probas_test = None
        self.cached_estimators = None

        if self.ag_predictor is not None:
            try:
                shutil.rmtree(self.ag_predictor.path, ignore_errors=True)
            except Exception:
                pass
            self.ag_predictor = None
        if self.ag_predictor_full is not None:
            try:
                shutil.rmtree(self.ag_predictor_full.path, ignore_errors=True)
            except Exception:
                pass
            self.ag_predictor_full = None

        gc.collect()

    # ─────────────────────────────────────────────────────────
    # Main run
    # ─────────────────────────────────────────────────────────
    def run(self, verbose=True):
        print(f"{'='*60}")
        print(f"DATASET: {self.name_file}  (seed={self.random_state})")
        print(f"{'='*60}")

        self.load_dataset()

        # 1. Train AG base models (L1 only) — this one must succeed
        self.train_autogluon_base_models()

        # 2. Evaluate individual L1 models
        self.evaluate_individual_l1_models()

        # Steps 3–7: each wrapped individually so a failure in one
        # (e.g., WeightedEnsemble depending on a broken XGBoost)
        # does not prevent the others from running.
        #
        # NOTE: AG full runs BEFORE SP random adaptive because the
        # adaptive time budget depends on actual AG full wall-clock time.
        steps = [
            ("WeightedEnsemble_L2",  self.evaluate_weighted_ensemble),
            ("SP greedy",            self.train_sp_greedy),
            ("SP random",            self.train_sp_random),
            ("AG full pipeline",     self.train_autogluon_full),
        ]
        if self.run_sp_random_adaptive:
            steps.append(("SP random adaptive", self.train_sp_random_adaptive))

        for step_name, step_fn in steps:
            try:
                step_fn()
            except Exception as e:
                print(f"  WARNING: {step_name} failed: {e}")

        print()


## Experiment Configuration

Adjust the parameters below before launching:
- `NRUNS`: number of repetitions with different seeds
- `AG_TIME_LIMIT`: seconds for AutoGluon L1 training per dataset/run
- `AG_FULL_TIME_LIMIT`: seconds for AutoGluon full pipeline per dataset/run
- `SP_RANDOM_TIME_LIMIT`: seconds per worker for SP random search (fixed budget)
- `RUN_SP_RANDOM_ADAPTIVE`: whether to run SP random with adaptive time = AG_full - AG_L1

**Methods compared** (all in one CSV):
1. **Best individual L1**: best single AG model selected on V
2. **WeightedEnsemble_L2**: AG learned linear aggregator over L1 models
3. **SP Greedy (partial)**: `allow_partial=True` — may use fewer than N models
4. **SP Greedy (full)**: `allow_partial=False` — always uses all N models
5. **SP Random**: time-limited random search over SP trees (fixed budget)
6. **AG Full Pipeline**: AutoGluon with stacking, bagging, best_quality preset
7. **SP Random Adaptive**: SP random with time budget = AG_full_time - AG_L1_time (optional, default on)

In [7]:
warnings.filterwarnings("ignore")

results = []
now = datetime.now()
NAME_FILE_CSV = 'results_multiclass_autogluon_' + now.strftime("%Y_%m_%d_%H_%M_%S") + ".csv"

# === Configuration ===
NRUNS = 10
AG_TIME_LIMIT = 120           # seconds for AG L1 models (60)
AG_FULL_TIME_LIMIT = 240      # seconds for AG full pipeline (120)
SP_RANDOM_TIME_LIMIT = 30     # seconds per worker for SP random search (12)
RUN_SP_RANDOM_ADAPTIVE = True # run SP random with adaptive time = AG_full - AG_L1

list_datasets = sorted(glob.glob("datasets_multiclass/*.csv"))
print(f"Found {len(list_datasets)} datasets")
print(list_datasets)

for run in range(NRUNS):
    for ds_path in list_datasets:
        try:
            d = dataset(
                ds_path,
                metric=lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro'),
                random_state=123 * (run + 1),
                ag_time_limit=AG_TIME_LIMIT,
                ag_full_time_limit=AG_FULL_TIME_LIMIT,
                sp_random_time_limit=SP_RANDOM_TIME_LIMIT,
                run_sp_random_adaptive=RUN_SP_RANDOM_ADAPTIVE,
            )
            d.run(verbose=True)
            d.dict_results['run'] = run
            d.dict_results['ag_time_limit'] = AG_TIME_LIMIT
            d.dict_results['ag_full_time_limit'] = AG_FULL_TIME_LIMIT
            d.dict_results['sp_random_time_limit'] = SP_RANDOM_TIME_LIMIT
            d.dict_results['run_sp_random_adaptive'] = RUN_SP_RANDOM_ADAPTIVE
            results.append(copy.deepcopy(d.dict_results))
            d.cleanup()
            del d
            gc.collect()
            # Save incrementally after every dataset
            pd.DataFrame(results).to_csv(NAME_FILE_CSV, index=False)
        except Exception as e:
            print(f"FAILED: {ds_path} run={run}")
            print(e)
            log_exception(e, ds_path, run, "exceptions.txt")

print(f"\nDone. Results saved to {NAME_FILE_CSV}")

Found 20 datasets
['datasets_multiclass/GesturePhaseSegmentationProcessed.csv', 'datasets_multiclass/Indian.csv', 'datasets_multiclass/JapaneseVowels.csv', 'datasets_multiclass/USPS.csv', 'datasets_multiclass/allbp.csv', 'datasets_multiclass/autoUniv.csv', 'datasets_multiclass/cjs.csv', 'datasets_multiclass/dna.csv', 'datasets_multiclass/eye.csv', 'datasets_multiclass/first.csv', 'datasets_multiclass/gas.csv', 'datasets_multiclass/jungle.csv', 'datasets_multiclass/led24.csv', 'datasets_multiclass/microaggregation2.csv', 'datasets_multiclass/page.csv', 'datasets_multiclass/segment.csv', 'datasets_multiclass/splice.csv', 'datasets_multiclass/texture.csv', 'datasets_multiclass/thyroid.csv', 'datasets_multiclass/wall.csv']
DATASET: datasets_multiclass/GesturePhaseSegmentationProcessed.csv  (seed=123)
  AutoGluon L1: 6 models in 122.1s
  Best L1: LightGBMXT (val=0.6484, test=0.6654)
  WeightedEnsemble_L2: val=0.6484, test=0.6654
  SP greedy (allow_partial=True): val=0.6484, test=0.6654, mod

2026-05-05 10:20:39,228	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


  AG full: val=0.6712, test=0.6903 (242.4s)
  SP random adaptive budget: 120.4s (AG_full=242.4s - AG_L1=122.1s)
  SP random adaptive: val=0.6477, test=0.6628, models=6, evaluated=2197744, time=123.2s

DATASET: datasets_multiclass/Indian.csv  (seed=123)
  AutoGluon L1: 6 models in 120.7s
  Best L1: NeuralNetFastAI (val=0.9458, test=0.9365)
  WeightedEnsemble_L2: val=0.9496, test=0.9215
  SP greedy (allow_partial=True): val=0.9483, test=0.9440, models=2, time=2.2s
  SP greedy (allow_partial=False): val=0.8680, test=0.9305, models=6, time=0.6s
  SP random: val=0.9498, test=0.9383, models=6, evaluated=131616, time=31.8s
  AG full: val=0.9386, test=0.9344 (239.6s)
  SP random adaptive budget: 118.9s (AG_full=239.6s - AG_L1=120.7s)
  SP random adaptive: val=0.9498, test=0.9383, models=6, evaluated=1947760, time=122.7s

DATASET: datasets_multiclass/JapaneseVowels.csv  (seed=123)
  AutoGluon L1: 6 models in 121.0s
  Best L1: NeuralNetFastAI (val=0.9887, test=0.9908)
  WeightedEnsemble_L2: val=

In [8]:
# Quick summary of results
try:
    df = pd.read_csv(NAME_FILE_CSV)
    print(f"Total rows: {len(df)}")
    print()

    methods = [
        ('best_l1_test_macro', 'Best individual L1'),
        ('we_l2_test_macro', 'WeightedEnsemble_L2'),
        ('sp_greedy_partial_test_macro', 'SP Greedy (partial)'),
        ('sp_greedy_full_test_macro', 'SP Greedy (full)'),
        ('sp_random_test_macro', 'SP Random (fixed)'),
        ('sp_random_adaptive_test_macro', 'SP Random (adaptive)'),
        ('ag_full_test_macro', 'AG Full Pipeline'),
    ]
    print("Mean F1-macro (test) across all datasets and runs:")
    print("-" * 50)
    for col, name in methods:
        if col in df.columns:
            mean_val = df[col].mean()
            print(f"  {name:30s} {mean_val:.6f}")

    print()
    print("Mean number of models used by SP methods:")
    print("-" * 50)
    for col, name in [('sp_greedy_partial_n_models_used', 'SP Greedy (partial)'),
                       ('sp_greedy_full_n_models_used', 'SP Greedy (full)'),
                       ('sp_random_n_models_used', 'SP Random (fixed)'),
                       ('sp_random_adaptive_n_models_used', 'SP Random (adaptive)')]:
        if col in df.columns:
            print(f"  {name:30s} {df[col].mean():.1f}")

    # Show adaptive time budget stats
    if 'sp_random_adaptive_time_budget' in df.columns:
        print()
        print("SP Random Adaptive time budget (seconds):")
        print("-" * 50)
        budget = df['sp_random_adaptive_time_budget'].dropna()
        if len(budget) > 0:
            print(f"  Mean:   {budget.mean():.1f}s")
            print(f"  Median: {budget.median():.1f}s")
            print(f"  Min:    {budget.min():.1f}s")
            print(f"  Max:    {budget.max():.1f}s")
except Exception as e:
    print(f"Could not load results: {e}")

Total rows: 200

Mean F1-macro (test) across all datasets and runs:
--------------------------------------------------
  Best individual L1             0.806427
  WeightedEnsemble_L2            0.808744
  SP Greedy (partial)            0.808900
  SP Greedy (full)               0.802190
  SP Random (fixed)              0.808272
  SP Random (adaptive)           0.807725
  AG Full Pipeline               0.815620

Mean number of models used by SP methods:
--------------------------------------------------
  SP Greedy (partial)            3.5
  SP Greedy (full)               9.2
  SP Random (fixed)              9.2
  SP Random (adaptive)           9.2

SP Random Adaptive time budget (seconds):
--------------------------------------------------
  Mean:   144.5s
  Median: 122.0s
  Min:    117.6s
  Max:    216.6s
